In [62]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import polars as pl
import numpy as np
from pathlib import Path

from torch.utils.data import Dataset, DataLoader, TensorDataset, ConcatDataset

In [63]:
# Check PyTorch version and CUDA support
import sys
print(f"Python executable: {sys.executable}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_built():
    print(f"MPS available: {torch.backends.mps.is_available()}")

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_built() and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"\nUsing {device} device")

Python executable: c:\Users\alexa\AppData\Local\Programs\Python\Python313\python.exe
PyTorch version: 2.10.0+cu130
CUDA available: True
CUDA version: 13.0
Number of GPUs: 1
GPU name: NVIDIA GeForce RTX 3060 Ti

Using cuda device


In [64]:
PREDICTIONS_H = 20
INPUT_LENGTH = 105
LEARNING_RATE = 1e-3  # Learning rate for Adam optimizer

In [65]:
class Encoder(nn.Module):
    """
    Bidirectional LSTM encoder that processes the input sequence.
    Uses 2 layers for better representation learning.
    """
    def __init__(self, input_dim, emb_dim, hidden_dim, num_layers=2, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, emb_dim)
        self.lstm = nn.LSTM(
            emb_dim, hidden_dim, 
            num_layers=num_layers,
            bidirectional=True, 
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

    def forward(self, x):
        # x: (batch, seq_len, input_dim)
        emb = F.relu(self.input_proj(x))  # (batch, seq_len, emb_dim)
        outputs, (h_n, c_n) = self.lstm(emb)
        # outputs: (batch, seq_len, hidden_dim * 2)
        # h_n: (num_layers * 2, batch, hidden_dim)
        return outputs, (h_n, c_n)


class Attention(nn.Module):
    """
    Bahdanau-style additive attention.
    Computes attention weights based on decoder hidden state and encoder outputs.
    """
    def __init__(self, encoder_dim, decoder_dim, attention_dim):
        super().__init__()
        self.encoder_proj = nn.Linear(encoder_dim, attention_dim)
        self.decoder_proj = nn.Linear(decoder_dim, attention_dim)
        self.v = nn.Linear(attention_dim, 1, bias=False)
    
    def forward(self, decoder_hidden, encoder_outputs):
        # decoder_hidden: (batch, decoder_dim)
        # encoder_outputs: (batch, seq_len, encoder_dim)
        
        # Project encoder outputs: (batch, seq_len, attention_dim)
        enc_proj = self.encoder_proj(encoder_outputs)
        # Project decoder hidden: (batch, 1, attention_dim)
        dec_proj = self.decoder_proj(decoder_hidden).unsqueeze(1)
        
        # Compute attention scores: (batch, seq_len, 1) -> (batch, seq_len)
        scores = self.v(torch.tanh(enc_proj + dec_proj)).squeeze(-1)
        
        # Softmax to get attention weights: (batch, seq_len)
        weights = F.softmax(scores, dim=1)
        
        # Weighted sum of encoder outputs: (batch, encoder_dim)
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)
        
        return context, weights


class Decoder(nn.Module):
    """
    Autoregressive decoder with attention.
    
    KEY DESIGN: Predicts ABSOLUTE values at each timestep (relative to start=0),
    NOT cumulative deltas. This prevents error accumulation that causes
    exponential drift in predictions.
    
    The hidden state is updated autoregressively for attention, but each
    output is an independent prediction of the value at that timestep.
    """
    def __init__(self, encoder_dim, hidden_dim, attention_dim, dropout=0.1):
        super().__init__()
        self.attention = Attention(encoder_dim, hidden_dim, attention_dim)
        # Input: previous prediction (1) embedded + timestep embedding
        self.input_embed = nn.Linear(1, 32)
        # Timestep embedding to help model know which step it's predicting
        self.time_embed = nn.Embedding(50, 16)  # Max 50 prediction steps
        self.lstm_cell = nn.LSTMCell(32 + 16 + encoder_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        # Output layer predicts ABSOLUTE value at this timestep (not delta)
        self.output = nn.Sequential(
            nn.Linear(hidden_dim + encoder_dim, hidden_dim),
            nn.Tanh(),  # Tanh to bound outputs, helps stability
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )
        self.hidden_dim = hidden_dim
    
    def forward(self, encoder_outputs, initial_hidden, initial_cell, num_steps, 
                start_value, teacher_targets=None, teacher_forcing_ratio=0.0):
        """
        Args:
            encoder_outputs: (batch, seq_len, encoder_dim)
            initial_hidden: (batch, hidden_dim)
            initial_cell: (batch, hidden_dim)
            num_steps: number of prediction steps
            start_value: (batch, 1) - starting point (typically 0)
            teacher_targets: (batch, num_steps) - ground truth for teacher forcing
            teacher_forcing_ratio: probability of using teacher forcing
        """
        batch_size = encoder_outputs.size(0)
        device = encoder_outputs.device
        
        predictions = []
        h, c = initial_hidden, initial_cell
        
        # Previous value for autoregressive input
        prev_value = start_value  # (batch, 1)
        
        for t in range(num_steps):
            # Embed the previous value
            value_emb = F.relu(self.input_embed(prev_value))  # (batch, 32)
            
            # Timestep embedding
            time_idx = torch.full((batch_size,), t, dtype=torch.long, device=device)
            time_emb = self.time_embed(time_idx)  # (batch, 16)
            
            # Compute attention context
            context, _ = self.attention(h, encoder_outputs)  # (batch, encoder_dim)
            
            # LSTM input: embedded value + time + context
            lstm_input = torch.cat([value_emb, time_emb, context], dim=1)
            
            # Update hidden state
            h, c = self.lstm_cell(lstm_input, (h, c))
            h_out = self.dropout(h) if self.training else h
            
            # Predict ABSOLUTE value at timestep t (not delta!)
            # This prevents error accumulation
            output_input = torch.cat([h_out, context], dim=1)
            pred = self.output(output_input)  # (batch, 1) - absolute prediction
            predictions.append(pred)
            
            # Decide next input: teacher forcing or own prediction
            if self.training and teacher_targets is not None and torch.rand(1).item() < teacher_forcing_ratio:
                prev_value = teacher_targets[:, t:t+1]
            else:
                # Use own prediction (detached during inference to save memory)
                prev_value = pred.detach() if not self.training else pred
        
        # Stack predictions: (batch, num_steps)
        predictions = torch.cat(predictions, dim=1)
        return predictions


class Seq2SeqModel(nn.Module):
    """
    Complete sequence-to-sequence model for time series prediction.
    
    Architecture:
    1. Encoder: Bidirectional LSTM processes input sequence
    2. Bridge: Projects encoder final state to decoder initial state
    3. Decoder: Autoregressive LSTM with attention, predicts one step at a time
    
    Key features:
    - Residual connection in decoder (predicts changes, not absolute values)
    - Attention mechanism for focusing on relevant input parts
    - Teacher forcing during training for faster convergence
    """
    def __init__(self, input_dim=5, emb_dim=64, hidden_dim=128, attention_dim=64, 
                 num_layers=2, dropout=0.1, pred_len=PREDICTIONS_H):
        super().__init__()
        self.encoder = Encoder(input_dim, emb_dim, hidden_dim, num_layers, dropout)
        encoder_dim = hidden_dim * 2  # Bidirectional
        
        # Bridge: convert encoder final states to decoder initial states
        self.h_bridge = nn.Linear(hidden_dim * 2 * num_layers, hidden_dim)
        self.c_bridge = nn.Linear(hidden_dim * 2 * num_layers, hidden_dim)
        
        self.decoder = Decoder(encoder_dim, hidden_dim, attention_dim, dropout)
        self.pred_len = pred_len
    
    def forward(self, x, teacher_targets=None, teacher_forcing_ratio=0.0):
        """
        Args:
            x: (batch, seq_len, input_dim) - input features
            teacher_targets: (batch, pred_len) - ground truth for teacher forcing
            teacher_forcing_ratio: probability of using teacher forcing
        
        Returns:
            predictions: (batch, pred_len) - predicted values
        """
        batch_size = x.size(0)
        
        # Encode input sequence
        encoder_outputs, (h_n, c_n) = self.encoder(x)
        # encoder_outputs: (batch, seq_len, hidden*2)
        # h_n: (num_layers*2, batch, hidden)
        
        # Flatten hidden states for bridge
        # h_n shape: (num_layers * 2, batch, hidden) -> (batch, num_layers * 2 * hidden)
        h_flat = h_n.permute(1, 0, 2).contiguous().view(batch_size, -1)
        c_flat = c_n.permute(1, 0, 2).contiguous().view(batch_size, -1)
        
        # Bridge to decoder initial states
        dec_h = torch.tanh(self.h_bridge(h_flat))  # (batch, hidden)
        dec_c = torch.tanh(self.c_bridge(c_flat))  # (batch, hidden)
        
        # Start value: the last price change in the input (first feature, last timestep)
        # Since targets are changes from last input price, start from 0
        start_value = torch.zeros(batch_size, 1, device=x.device)
        
        # Decode
        predictions = self.decoder(
            encoder_outputs, dec_h, dec_c, 
            self.pred_len, start_value,
            teacher_targets, teacher_forcing_ratio
        )
        
        return predictions

In [66]:
class StockDataset(Dataset):
    """
    Dataset that uses per-window normalization and predicts CHANGES from the last input price.
    This is critical for getting non-flat predictions.
    
    Key insight: Predicting absolute prices across different stocks fails because
    the model learns to predict the mean. Predicting percentage changes normalizes
    across stocks and forces the model to learn temporal dynamics.
    """
    def __init__(self, name, df):
        self.input_len = INPUT_LENGTH
        self.pred_len = PREDICTIONS_H
        self.name = name

        # Store raw prices for per-window normalization
        self.prices = torch.tensor(df["Close"].to_numpy(), dtype=torch.float32)
        self.rsi = torch.tensor(df["Rsi"].to_numpy(), dtype=torch.float32) / 100
        self.k = torch.tensor(df["%K"].to_numpy(), dtype=torch.float32) / 100
        self.d = torch.tensor(df["%D"].to_numpy(), dtype=torch.float32) / 100
        dividends = df["Dividends"].to_numpy() if "Dividends" in df.columns else np.zeros(len(df), dtype=np.float32)
        self.dividends = torch.tensor(dividends, dtype=torch.float32)

    def __len__(self):
        return len(self.prices) - self.input_len - self.pred_len
    
    def __repr__(self):
        return self.name + f" of len {self.__len__()}"
    
    def __getitem__(self, idx):
        # Get the raw price window
        input_prices = self.prices[idx:idx + self.input_len]
        target_prices = self.prices[idx + self.input_len:idx + self.input_len + self.pred_len]
        
        # Per-window normalization: use the FIRST price in the window as reference
        # This makes all windows comparable regardless of absolute price level
        ref_price = input_prices[0]
        
        # Normalize prices as percentage change from reference (window start)
        # This gives values typically in range [-0.5, 0.5] for most stocks
        input_price_norm = (input_prices - ref_price) / (ref_price + 1e-8)
        
        # Target: percentage change from the LAST INPUT PRICE (not window start)
        # This is critical - model predicts how much price will change from current
        last_input_price = input_prices[-1]
        target_changes = (target_prices - last_input_price) / (last_input_price + 1e-8)
        
        # Get other features for this window
        rsi = self.rsi[idx:idx + self.input_len]
        k = self.k[idx:idx + self.input_len]
        d = self.d[idx:idx + self.input_len]
        divs = self.dividends[idx:idx + self.input_len]
        
        # Stack all features: (seq_len, num_features)
        x = torch.stack([input_price_norm, rsi, k, d, divs], dim=1)
        y = target_changes
        
        return x, y


In [67]:
SPLIT_RATIO = 0.7
data_dir = Path("stocks")
files = list(data_dir.glob("*/*/*.parquet")) or list(data_dir.glob("*/*/*.csv"))
train_datasets = []
val_datasets = []

for file in files:
    name = file.stem
    df = pl.read_parquet(file) if file.suffix == ".parquet" else pl.read_csv(file)
    split_idx = int(len(df) * SPLIT_RATIO)

    train_df = df.head(split_idx)
    validation_df = df.tail(len(df) - split_idx)

    train_set = StockDataset(name, train_df)
    val_set = StockDataset(name, validation_df)

    if len(train_set) > 0:
        train_datasets.append(train_set)
    if len(val_set) > 0:
        val_datasets.append(val_set)

In [68]:
train_ds = ConcatDataset(train_datasets) if len(train_datasets) > 0 else None
val_ds = ConcatDataset(val_datasets) if len(val_datasets) > 0 else None

batch_size = 32
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True) if train_ds is not None else None
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False) if val_ds is not None else None

In [69]:
print(f"Number of training datasets: {len(train_datasets)}")
print(f"Number of validation datasets: {len(val_datasets)}")
if train_ds is not None:
    print(f"Total training samples: {len(train_ds)}")
if val_ds is not None:
    print(f"Total validation samples: {len(val_ds)}")

if train_loader is not None:
    sample_x, sample_y = next(iter(train_loader))
    print(f"\nSample batch shapes:")
    print(f"  X shape: {sample_x.shape}")
    print(f"  Y shape: {sample_y.shape}")


Number of training datasets: 25
Number of validation datasets: 25
Total training samples: 17166
Total validation samples: 5580

Sample batch shapes:
  X shape: torch.Size([32, 105, 5])
  Y shape: torch.Size([32, 20])


In [70]:
def init_weights(m):
    """Initialize weights for better training stability."""
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, (nn.LSTM, nn.LSTMCell)):
        for name, param in m.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param.data)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param.data)
            elif 'bias' in name:
                param.data.fill_(0)
                # Set forget gate bias to 1 for better gradient flow
                n = param.size(0)
                param.data[n // 4:n // 2].fill_(1)


class PredictionLoss(nn.Module):
    """
    Custom loss function for time series prediction.
    
    Components:
    1. Huber Loss - Main accuracy metric (robust to outliers)
    2. Variance Loss - Penalizes if predictions have less variance than targets
    3. Smoothness Loss - Penalizes overly erratic predictions (prevents noise)
    4. Trend Loss - Rewards matching overall trend direction
    """
    def __init__(self, var_weight=0.05, smooth_weight=0.02, trend_weight=0.1):
        super().__init__()
        self.var_weight = var_weight
        self.smooth_weight = smooth_weight
        self.trend_weight = trend_weight
        self.huber = nn.HuberLoss(delta=0.1)
    
    def forward(self, preds, targets):
        # 1. Huber Loss (main component, robust to outliers)
        huber_loss = self.huber(preds, targets)
        
        # 2. Variance Loss - encourage predictions to have similar variance to targets
        pred_var = preds.var(dim=1).mean()
        target_var = targets.var(dim=1).mean()
        # Penalize if prediction variance is much lower than target variance
        var_loss = F.relu(target_var - pred_var)
        
        # 3. Smoothness Loss - penalize overly large jumps between consecutive predictions
        # This prevents noisy/erratic predictions without forcing flatness
        if preds.size(1) > 1:
            pred_diff = preds[:, 1:] - preds[:, :-1]
            target_diff = targets[:, 1:] - targets[:, :-1]
            # Penalize if prediction changes are much larger than target changes
            smooth_loss = F.relu(pred_diff.abs() - target_diff.abs() * 1.5).mean()
        else:
            smooth_loss = torch.tensor(0.0, device=preds.device)
        
        # 4. Trend Loss - reward matching overall trend direction
        # Compare start-to-end direction
        pred_trend = preds[:, -1] - preds[:, 0]  # (batch,)
        target_trend = targets[:, -1] - targets[:, 0]
        # Sign agreement: reward when both positive or both negative
        trend_agreement = (pred_trend * target_trend).clamp(min=-1, max=1)
        trend_loss = (1 - trend_agreement.mean()) * 0.5  # Scale to [0, 1]
        
        # Combine losses
        total_loss = huber_loss + self.var_weight * var_loss + self.smooth_weight * smooth_loss + self.trend_weight * trend_loss
        
        return total_loss, huber_loss, var_loss, trend_loss

def train_and_validate(model, train_loader, val_loader, epochs=100, save_path="model.pt",
                        patience=15, teacher_forcing_start=0.8, teacher_forcing_end=0.2):
    """
    Train the model with scheduled teacher forcing and custom loss.
    
    Key features:
    - Custom loss that penalizes flat predictions
    - Scheduled teacher forcing (decreases over training)
    - Early stopping based on validation loss
    - Learning rate scheduling
    """
    import os
    os.makedirs("models", exist_ok=True)
    
    model.to(device)
    model.apply(init_weights)
    
    # Optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
    
    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6
    )
    
    # Custom loss function
    criterion = PredictionLoss(var_weight=0.05, smooth_weight=0.02, trend_weight=0.1)
    
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    patience_counter = 0
    best_epoch = 0
    
    for epoch in range(1, epochs + 1):
        # Calculate current teacher forcing ratio (linear decay)
        tf_ratio = teacher_forcing_start - (teacher_forcing_start - teacher_forcing_end) * (epoch - 1) / max(epochs - 1, 1)
        
        # Training phase
        model.train()
        train_loss = 0.0
        train_huber = 0.0
        train_n = 0
        
        for x, y in train_loader:
            x = x.to(device).float()
            y = y.to(device).float()
            
            # Skip batches with invalid data
            if torch.isnan(x).any() or torch.isnan(y).any():
                continue
            
            optimizer.zero_grad()
            
            # Forward pass with teacher forcing
            preds = model(x, teacher_targets=y, teacher_forcing_ratio=tf_ratio)
            
            # Compute loss
            loss, huber, var_loss, trend_loss = criterion(preds, y)
            
            if not torch.isfinite(loss):
                continue
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
            train_huber += huber.item()
            train_n += 1
        
        avg_train_loss = train_loss / train_n if train_n > 0 else float('nan')
        avg_train_huber = train_huber / train_n if train_n > 0 else float('nan')
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_huber = 0.0
        val_n = 0
        
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device).float()
                y = y.to(device).float()
                
                if torch.isnan(x).any() or torch.isnan(y).any():
                    continue
                
                # Forward pass without teacher forcing
                preds = model(x, teacher_forcing_ratio=0.0)
                
                loss, huber, _, _ = criterion(preds, y)
                
                if not torch.isfinite(loss):
                    continue
                
                val_loss += loss.item()
                val_huber += huber.item()
                val_n += 1
        
        avg_val_loss = val_loss / val_n if val_n > 0 else float('nan')
        avg_val_huber = val_huber / val_n if val_n > 0 else float('nan')
        
        train_losses.append(avg_train_huber)
        val_losses.append(avg_val_huber)
        
        # Learning rate scheduling
        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(avg_val_loss)
        new_lr = optimizer.param_groups[0]['lr']
        
        # Early stopping check
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_epoch = epoch
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
            print(f"✓ Epoch {epoch:3d} | Train: {avg_train_huber:.6f} | Val: {avg_val_huber:.6f} | TF: {tf_ratio:.2f} | LR: {new_lr:.2e} | BEST")
        else:
            patience_counter += 1
            print(f"  Epoch {epoch:3d} | Train: {avg_train_huber:.6f} | Val: {avg_val_huber:.6f} | TF: {tf_ratio:.2f} | LR: {new_lr:.2e} | Patience: {patience_counter}/{patience}")
        
        if new_lr < old_lr:
            print(f"  → Learning rate reduced to {new_lr:.2e}")
        
        if patience_counter >= patience:
            print(f"\n⏹ Early stopping at epoch {epoch}. Best: {best_val_loss:.6f} at epoch {best_epoch}")
            break
        
        # Save periodic checkpoints
        if epoch % 20 == 0:
            torch.save(model.state_dict(), f"models/model_epoch_{epoch}.pt")
    
    print(f"\nTraining complete. Best val loss: {best_val_loss:.6f} at epoch {best_epoch}")
    return save_path, train_losses, val_losses

In [71]:
# Create model with new architecture
# Key: predicting price CHANGES with residual connections in decoder
model = Seq2SeqModel(
    input_dim=5,       # 5 features: price_change, rsi, stoch_k, stoch_d, dividends
    emb_dim=64,        # Input embedding dimension
    hidden_dim=128,    # LSTM hidden dimension
    attention_dim=64,  # Attention projection dimension
    num_layers=2,      # Number of LSTM layers
    dropout=0.1,       # Dropout for regularization
    pred_len=PREDICTIONS_H
)

save_path = "model.pt"
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Train with scheduled teacher forcing and custom loss
save_path_used, train_history, val_history = train_and_validate(
    model,
    train_loader,
    val_loader,
    epochs=100,
    save_path=save_path,
    patience=15,
    teacher_forcing_start=0.8,  # Start with 80% teacher forcing
    teacher_forcing_end=0.2     # End with 20% teacher forcing
)

Model parameters: 1,022,881
✓ Epoch   1 | Train: 0.003563 | Val: 0.001153 | TF: 0.80 | LR: 1.00e-03 | BEST
  Epoch   2 | Train: 0.000526 | Val: 0.001600 | TF: 0.79 | LR: 1.00e-03 | Patience: 1/15
  Epoch   3 | Train: 0.000539 | Val: 0.001440 | TF: 0.79 | LR: 1.00e-03 | Patience: 2/15
  Epoch   4 | Train: 0.000516 | Val: 0.002992 | TF: 0.78 | LR: 1.00e-03 | Patience: 3/15
  Epoch   5 | Train: 0.000669 | Val: 0.001525 | TF: 0.78 | LR: 1.00e-03 | Patience: 4/15
  Epoch   6 | Train: 0.000756 | Val: 0.002091 | TF: 0.77 | LR: 1.00e-03 | Patience: 5/15
  Epoch   7 | Train: 0.000894 | Val: 0.001343 | TF: 0.76 | LR: 5.00e-04 | Patience: 6/15
  → Learning rate reduced to 5.00e-04
✓ Epoch   8 | Train: 0.000964 | Val: 0.001120 | TF: 0.76 | LR: 5.00e-04 | BEST
  Epoch   9 | Train: 0.001023 | Val: 0.001202 | TF: 0.75 | LR: 5.00e-04 | Patience: 1/15
  Epoch  10 | Train: 0.001082 | Val: 0.001139 | TF: 0.75 | LR: 5.00e-04 | Patience: 2/15
  Epoch  11 | Train: 0.001155 | Val: 0.001372 | TF: 0.74 | LR: 5

In [72]:
import altair as alt

history_df = pl.DataFrame({
    'Epoch': list(range(1, len(train_history) + 1)) * 2,
    'Loss': train_history + val_history,
    'Type': ['Train'] * len(train_history) + ['Validation'] * len(val_history)
})

chart = alt.Chart(history_df.to_pandas()).mark_line(point=True).encode(
    x=alt.X('Epoch:Q', title='Epoch'),
    y=alt.Y('Loss:Q', title='Huber Loss'),
    color=alt.Color('Type:N', legend=alt.Legend(title='Type'))
).properties(
    width=600,
    height=300,
    title='Training and Validation Loss Over Time'
).configure_axis(
    gridOpacity=0.3
)

chart.save('training_history.png', scale_factor=2)
chart


alt.Chart(...)

In [73]:
import altair as alt

# Load the best model weights
model.load_state_dict(torch.load(save_path_used if 'save_path_used' in locals() else "model.pt", weights_only=True))
model.eval()

with torch.no_grad():
    # Get multiple batches for better evaluation
    all_preds = []
    all_targets = []
    
    for i, (x_batch, y_batch) in enumerate(val_loader):
        if i >= 5:  # Use first 5 batches
            break
        x_batch = x_batch.to(device).float()
        preds = model(x_batch, teacher_forcing_ratio=0.0)
        all_preds.append(preds.cpu())
        all_targets.append(y_batch)
    
    preds_all = torch.cat(all_preds, dim=0).numpy()
    targets_all = torch.cat(all_targets, dim=0).numpy()
    
    # Calculate metrics
    mae = np.mean(np.abs(preds_all - targets_all))
    mse = np.mean((preds_all - targets_all) ** 2)
    rmse = np.sqrt(mse)
    
    # Check prediction variance vs target variance
    pred_var = np.var(preds_all, axis=1).mean()
    target_var = np.var(targets_all, axis=1).mean()
    
    print(f"\n{'='*60}")
    print(f"MODEL PERFORMANCE METRICS")
    print(f"{'='*60}")
    print(f"Mean Absolute Error (MAE):  {mae:.6f}")
    print(f"Mean Squared Error (MSE):   {mse:.6f}")
    print(f"Root Mean Squared Error:    {rmse:.6f}")
    print(f"Prediction Variance:        {pred_var:.6f}")
    print(f"Target Variance:            {target_var:.6f}")
    print(f"Variance Ratio:             {pred_var/target_var:.2%}")
    print(f"{'='*60}")
    
    # Plot first 6 samples
    time_steps = np.arange(PREDICTIONS_H)
    plot_rows = []
    for i in range(min(6, len(preds_all))):
        plot_rows.append(pl.DataFrame({
            'Time Step': list(time_steps) * 2,
            'Price Change (%)': list(targets_all[i] * 100) + list(preds_all[i] * 100),  # Convert to percentage
            'Type': ['Actual'] * PREDICTIONS_H + ['Predicted'] * PREDICTIONS_H,
            'Sample': [f'Sample {i+1}'] * (PREDICTIONS_H * 2)
        }))
    df_plot = pl.concat(plot_rows).to_pandas()
    
    chart = alt.Chart(df_plot).mark_line(point=True).encode(
        x=alt.X('Time Step:Q', title='Prediction Horizon'),
        y=alt.Y('Price Change (%):Q', title='Price Change from Last Input (%)'),
        color=alt.Color('Type:N', legend=alt.Legend(title='Type')),
        column=alt.Column('Sample:N', header=alt.Header(title='Actual vs Predicted Price Changes'))
    ).properties(
        width=200,
        height=150
    ).configure_axis(
        gridOpacity=0.3
    )
    
    chart.save('predictions.png', scale_factor=2)
    chart



MODEL PERFORMANCE METRICS
Mean Absolute Error (MAE):  0.025072
Mean Squared Error (MSE):   0.001069
Root Mean Squared Error:    0.032694
Prediction Variance:        0.000025
Target Variance:            0.000333
Variance Ratio:             7.46%
